# Plant Disease Detection - Kaggle Notebook

**Author:** Farrel Ghozy Affifudin (452024611053)
**Course:** Pembelajaran Mesin 2
**Topic:** Real-Time Plant Disease Detection Using Lightweight Deep Learning with Transfer Learning

## Structure
1. Setup & Imports
2. Data Loading & Preprocessing
3. Baseline Model (Custom CNN)
4. Deep Learning Model (MobileNetV3-Small)
5. Experimental Scenarios
   - Scenario 1: MobileNetV3-Small (Transfer Learning)
   - Scenario 2: MobileNetV3-Small + Data Augmentation
   - Scenario 3: MobileNetV3-Small + Fine-tuning
6. Comparative Analysis
7. Error Analysis
8. Conclusion

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, regularizers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Data Loading & Preprocessing

In [ ]:
# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30

print("Configuration:")
print(f"  - Image size: {IMG_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Epochs: {EPOCHS}")

In [ ]:
# Check if dataset exists
DATASET_PATH = Path('/kaggle/input/plantvillage-dataset/plantvillage-dataset')

if not DATASET_PATH.exists():
    print("Dataset not found! Please upload PlantVillage dataset to /kaggle/input/plantvillage-dataset/")
    print("Expected structure:")
    print("  /kaggle/input/plantvillage-dataset/")
    print("    ├── tomato___healthy/ (and other classes)")
    print("    └── ...")
else:
    print("Dataset found!")
    print(f"Path: {DATASET_PATH}")
    print(f"Contents: {list(DATASET_PATH.iterdir())[:10]}")

In [ ]:
# Load dataset using ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED
)

print("\nClass indices:", train_generator.class_indices)
print("Number of classes:", len(train_generator.class_indices))
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)

In [ ]:
# Analyze class distribution
train_labels = train_generator.classes
val_labels = val_generator.classes

class_names = list(train_generator.class_indices.keys())

train_class_counts = pd.Series(train_labels).value_counts().sort_index()
val_class_counts = pd.Series(val_labels).value_counts().sort_index()

print("\n=== Training Class Distribution ===")
print(train_class_counts)
print("\n=== Validation Class Distribution ===")
print(val_class_counts)

# Check for class imbalance
max_train = train_class_counts.max()
min_train = train_class_counts.min()
imbalance_ratio_train = max_train / min_train

max_val = val_class_counts.max()
min_val = val_class_counts.min()
imbalance_ratio_val = max_val / min_val

print(f"\nTraining imbalance ratio: {imbalance_ratio_train:.2f}")
print(f"Validation imbalance ratio: {imbalance_ratio_val:.2f}")

## 3. Baseline Model (Custom CNN)

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=38):
    """
    Build a simple CNN baseline model.
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

custom_cnn = build_custom_cnn(num_classes=len(class_names))
custom_cnn.summary()

In [ ]:
# Compile Custom CNN
custom_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Custom CNN compiled successfully!")

In [ ]:
# Callbacks for Custom CNN
checkpoint_path = '/kaggle/working/custom_cnn_best.h5'

callbacks_custom = [
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# Train Custom CNN
print("Training Custom CNN baseline...")
history_custom = custom_cnn.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_custom,
    verbose=1
)

In [ ]:
# Evaluate Custom CNN
custom_cnn.load_weights(checkpoint_path)

val_loss, val_acc = custom_cnn.evaluate(val_generator, verbose=0)
print(f"\nCustom CNN - Validation Accuracy: {val_acc:.4f}")
print(f"Custom CNN - Validation Loss: {val_loss:.4f}")

## 4. Deep Learning Model (MobileNetV3-Small)

In [ ]:
def build_mobilenetv3(num_classes=38, trainable=False):
    """
    Build MobileNetV3-Small with transfer learning.
    trainable: False = freeze all layers, True = fine-tune all layers
    """
    base_model = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    
    base_model.trainable = trainable
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    predictions = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=base_model.input, outputs=predictions)
    
    return model

mobilenetv3 = build_mobilenetv3(num_classes=len(class_names), trainable=False)
mobilenetv3.summary()

In [ ]:
# Compile MobileNetV3
mobilenetv3.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("MobileNetV3 compiled successfully!")

In [ ]:
# Callbacks for MobileNetV3
mobilenet_checkpoint_path = '/kaggle/working/mobilenetv3_small_best.h5'

callbacks_mobilenet = [
    callbacks.ModelCheckpoint(
        filepath=mobilenet_checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# Train MobileNetV3
print("Training MobileNetV3-Small...")
history_mobilenet = mobilenetv3.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_mobilenet,
    verbose=1
)

In [ ]:
# Evaluate MobileNetV3
mobilenetv3.load_weights(mobilenet_checkpoint_path)

val_loss, val_acc = mobilenetv3.evaluate(val_generator, verbose=0)
print(f"\nMobileNetV3-Small - Validation Accuracy: {val_acc:.4f}")
print(f"MobileNetV3-Small - Validation Loss: {val_loss:.4f}")

## 5. Experimental Scenarios

### Scenario 1: MobileNetV3-Small (Transfer Learning) - ALREADY DONE

In [ ]:
# Display training history for Scenario 1
def plot_training_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].plot(history.history['accuracy'], label='Training Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history.history['loss'], label='Training Loss')
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history_custom, "Custom CNN (Baseline)")
plot_training_history(history_mobilenet, "MobileNetV3-Small (Scenario 1)")

### Scenario 2: MobileNetV3-Small + Data Augmentation

In [ ]:
# Build model with data augmentation
def build_mobilenetv3_augmentation(num_classes=38):
    datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=0.2
    )
    
    train_gen = datagen.flow_from_directory(
        DATASET_PATH,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        seed=SEED
    )
    
    val_gen = datagen.flow_from_directory(
        DATASET_PATH,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        seed=SEED
    )
    
    base_model = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    predictions = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=base_model.input, outputs=predictions)
    
    return model, train_gen, val_gen

mobilenetv3_aug, train_gen_aug, val_gen_aug = build_mobilenetv3_augmentation(len(class_names))
mobilenetv3_aug.summary()

In [ ]:
# Compile with lower learning rate
mobilenetv3_aug.compile(
    optimizer=optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Callbacks for Scenario 2
aug_checkpoint_path = '/kaggle/working/mobilenetv3_augmentation_best.h5'

callbacks_aug = [
    callbacks.ModelCheckpoint(
        filepath=aug_checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# Train Scenario 2
print("Training MobileNetV3-Small + Augmentation...")
history_aug = mobilenetv3_aug.fit(
    train_gen_aug,
    epochs=EPOCHS,
    validation_data=val_gen_aug,
    callbacks=callbacks_aug,
    verbose=1
)

In [ ]:
# Evaluate Scenario 2
mobilenetv3_aug.load_weights(aug_checkpoint_path)

val_loss_aug, val_acc_aug = mobilenetv3_aug.evaluate(val_gen_aug, verbose=0)
print(f"\nMobileNetV3-Small + Aug - Val Acc: {val_acc_aug:.4f}")
print(f"MobileNetV3-Small + Aug - Val Loss: {val_loss_aug:.4f}")

### Scenario 3: MobileNetV3-Small + Fine-tuning

In [ ]:
# Build model for fine-tuning
def build_mobilenetv3_finetuning(num_classes=38):
    base_model = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    predictions = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=base_model.input, outputs=predictions)
    
    return model, base_model

mobilenetv3_finetune, base_model_ft = build_mobilenetv3_finetuning(len(class_names))
mobilenetv3_finetune.summary()

In [ ]:
# Compile with low learning rate
mobilenetv3_finetune.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Callbacks for Scenario 3
finetune_checkpoint_path = '/kaggle/working/mobilenetv3_finetune_best.h5'

callbacks_finetune = [
    callbacks.ModelCheckpoint(
        filepath=finetune_checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# Phase 1: Train head only
print("Phase 1: Training head only...")
history_finetune_phase1 = mobilenetv3_finetune.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_finetune,
    verbose=1
)

In [ ]:
# Phase 2: Fine-tuning
print("\nPhase 2: Fine-tuning...")
base_model_ft.trainable = True

for layer in base_model_ft.layers[:-20]:
    layer.trainable = False

mobilenetv3_finetune.compile(
    optimizer=optimizers.Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune_phase2 = mobilenetv3_finetune.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_finetune,
    verbose=1
)

In [ ]:
# Evaluate Scenario 3
mobilenetv3_finetune.load_weights(finetune_checkpoint_path)

val_loss_ft, val_acc_ft = mobilenetv3_finetune.evaluate(val_generator, verbose=0)
print(f"\nMobileNetV3-Small + Fine-tune - Val Acc: {val_acc_ft:.4f}")
print(f"MobileNetV3-Small + Fine-tune - Val Loss: {val_loss_ft:.4f}")

## 6. Comparative Analysis

In [ ]:
# Comparison table
results = {
    'Model': ['Custom CNN (Baseline)', 'MobileNetV3-Small (Scenario 1)',
              'MobileNetV3-Small + Aug (Scenario 2)', 'MobileNetV3-Small + Fine-tune (Scenario 3)'],
    'Validation Accuracy': [val_acc, val_acc, val_acc_aug, val_acc_ft],
    'Validation Loss': [val_loss, val_loss, val_loss_aug, val_loss_ft],
    'Parameters (M)': ['~3.5M', '~2.5M', '~2.5M', '~2.5M']
}

results_df = pd.DataFrame(results)
print("\n=== Experimental Results ===")
print(results_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(results['Model'], results['Validation Accuracy'], color='skyblue')
axes[0].set_title('Validation Accuracy')
axes[0].set_ylim([0.7, 1.0])
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(results['Model'], results['Validation Loss'], color='salmon')
axes[1].set_title('Validation Loss')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
# Get predictions
val_images, val_labels = next(iter(val_generator))
val_preds = mobilenetv3.predict(val_images[:100])
val_pred_classes = np.argmax(val_preds, axis=1)
val_true_classes = np.argmax(val_labels[:100], axis=1)

# Confusion matrix
conf_matrix = confusion_matrix(val_true_classes, val_pred_classes)

plt.figure(figsize=(20, 18))
sns.heatmap(conf_matrix, annot=False, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Conclusion

In [ ]:
# Summary
print("=== EXPERIMENTAL SUMMARY ===")
print(f"1. Baseline: Custom CNN ({val_acc:.4f})")
print(f"2. Scenario 1 (Transfer): {val_acc:.4f}")
print(f"3. Scenario 2 (Aug): {val_acc_aug:.4f}")
print(f"4. Scenario 3 (Fine-tune): {val_acc_ft:.4f}")

best_model = max([val_acc, val_acc_aug, val_acc_ft], key=lambda x: x)
if best_model == val_acc:
    best_scenario = "Scenario 1 (Transfer Learning)"
elif best_model == val_acc_aug:
    best_scenario = "Scenario 2 (Transfer + Augmentation)"
else:
    best_scenario = "Scenario 3 (Transfer + Fine-tuning)"
print(f"\nBest Model: {best_scenario} ({best_model:.4f})")

## End of Notebook